# Projet MLOps — Classification exploratoire du risque de diabète

**Objectif pédagogique :** démontrer un pipeline reproductible et son suivi, et non fournir un diagnostic médical.

Dataset fourni : `data/diabetes_risk.csv`. Les résultats doivent être réexécutés par l'équipe avant la remise finale.

## 1. Imports et configuration

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
DATA_PATH = Path("../data/diabetes_risk.csv")  # si notebook ouvert depuis notebooks/
if not DATA_PATH.exists():
    DATA_PATH = Path("data/diabetes_risk.csv")
df = pd.read_csv(DATA_PATH)
df.shape

## 2. Compréhension des données et qualité

In [ ]:
display(df.head())
print(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("valeurs_manquantes"))
display(df["diabetes_risk"].value_counts().to_frame("effectif"))


## 3. Séparation des variables
`patient_id` est retiré des prédicteurs. La séparation est stratifiée pour conserver la proportion des classes.

In [ ]:
TARGET = "diabetes_risk"
X = df.drop(columns=[TARGET, "patient_id"])
y = df[TARGET]
numeric = X.select_dtypes(include=["number"]).columns.tolist()
categorical = X.select_dtypes(exclude=["number"]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 4. Pipeline de préparation et modèle

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250, random_state=RANDOM_STATE,
        class_weight="balanced", n_jobs=-1, min_samples_leaf=2
    ))
])
model.fit(X_train, y_train)


## 5. Évaluation sur le jeu de test

In [ ]:
pred = model.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    "macro_f1": f1_score(y_test, pred, average="macro")
}
metrics


In [ ]:
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=["Low", "Moderate", "High"], cmap=None)
plt.title("Matrice de confusion")
plt.tight_layout()
plt.show()


## 6. Importances internes du modèle
Les importances décrivent la contribution utilisée par le modèle, pas une causalité.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importance = model.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({"variable": feature_names, "importance": importance}).sort_values("importance", ascending=False)
display(importance_df.head(15))


## 7. Suivi MLflow (facultatif dans le notebook)

Installer MLflow avec `pip install mlflow`, puis journaliser paramètres et métriques. Le script `src/train.py` réalise cette étape automatiquement lorsqu'il est exécuté dans un environnement où MLflow est installé.

```python
import mlflow
mlflow.set_experiment("diabetes-risk-classification")
with mlflow.start_run():
    mlflow.log_param("algorithm", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 250)
    mlflow.log_metric("accuracy", metrics["accuracy"])
    mlflow.log_metric("balanced_accuracy", metrics["balanced_accuracy"])
    mlflow.log_metric("macro_f1", metrics["macro_f1"])
```

## 8. Interprétation et limites
- Évaluer le rappel de la classe High et les confusions entre Moderate et les autres classes.
- Vérifier la documentation source de `diabetes_risk` : si la cible est construite à partir de HbA1c ou de glycémie, les scores peuvent refléter cette règle de construction.
- Ces résultats sont exploratoires, ne sont pas une validation clinique et ne doivent pas servir à une décision médicale.